In [ ]:
%pip install -q --upgrade pip
%pip install -q ultralytics fiftyone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import urllib.request

ANNOT_URL = "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv"
CLASS_URL  = "https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv"

os.makedirs("/tmp/oi_meta", exist_ok=True)
annot_path = "/tmp/oi_meta/train_bbox.csv"
class_path  = "/tmp/oi_meta/classes.csv"

if not os.path.exists(annot_path):
    print("Downloading annotations (~1.5 GB)...")
    urllib.request.urlretrieve(ANNOT_URL, annot_path)

if not os.path.exists(class_path):
    urllib.request.urlretrieve(CLASS_URL, class_path)

annot      = pd.read_csv(annot_path, usecols=["LabelName"])
classes_df = pd.read_csv(class_path, header=None, names=["LabelName", "DisplayName"])

top100_labels  = annot["LabelName"].value_counts().head(100).index.tolist()
label_to_name  = dict(zip(classes_df["LabelName"], classes_df["DisplayName"]))
TOP100_CLASSES = [label_to_name[l] for l in top100_labels if l in label_to_name]

print(f"Top-100 classes ({len(TOP100_CLASSES)}):")
for i, c in enumerate(TOP100_CLASSES):
    print(f"  {i}: {c}")

Top-100 classes (100):
  0: Clothing
  1: Man
  2: Tree
  3: Human face
  4: Person
  5: Woman
  6: Footwear
  7: Window
  8: Flower
  9: Wheel
  10: Plant
  11: Car
  12: Human hair
  13: Human arm
  14: Human head
  15: Girl
  16: Building
  17: Human body
  18: Mammal
  19: House
  20: Chair
  21: Tire
  22: Suit
  23: Fashion accessory
  24: Food
  25: Boy
  26: Table
  27: Skyscraper
  28: Land vehicle
  29: Boat
  30: Jeans
  31: Human eye
  32: Human hand
  33: Human leg
  34: Toy
  35: Tower
  36: Human nose
  37: Bicycle wheel
  38: Glasses
  39: Dress
  40: Vehicle
  41: Bird
  42: Sports equipment
  43: Street light
  44: Human mouth
  45: Palm tree
  46: Book
  47: Tableware
  48: Drink
  49: Bottle
  50: Bicycle
  51: Furniture
  52: Snack
  53: Sculpture
  54: Flag
  55: Dog
  56: Dessert
  57: Microphone
  58: Fruit
  59: Jacket
  60: Guitar
  61: Fast food
  62: Drum
  63: Sunglasses
  64: Poster
  65: Fish
  66: Baked goods
  67: Shelf
  68: Houseplant
  69: Flowerpot


In [ ]:
import shutil
import fiftyone as fo
import fiftyone.zoo as foz

ROOT       = "/content/drive/MyDrive/yolo_top100_rpi"
EXPORT_DIR = os.path.join(ROOT, "dataset")
os.makedirs(ROOT, exist_ok=True)

if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

train_ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=TOP100_CLASSES,
    only_matching=True,
    shuffle=True,
    seed=42,
    max_samples=12000,
)

val_ds = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=TOP100_CLASSES,
    only_matching=True,
    shuffle=True,
    seed=42,
    max_samples=2000,
)

print(f"Train: {len(train_ds)} images")
print(f"Val:   {len(val_ds)} images")

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv' to '/root/fiftyone/open-images-v7/train/metadata/image_ids.csv'


 100% |██████|    4.8Gb/4.8Gb [2.3s elapsed, 0s remaining, 2.2Gb/s]        


INFO:eta.core.utils: 100% |██████|    4.8Gb/4.8Gb [2.3s elapsed, 0s remaining, 2.2Gb/s]        


...

 100% |███████████████| 2000/2000 [16.6s elapsed, 0s remaining, 121.1 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 2000/2000 [16.6s elapsed, 0s remaining, 121.1 samples/s]      


Dataset 'open-images-v7-validation-2000' created


INFO:fiftyone.zoo.datasets:Dataset 'open-images-v7-validation-2000' created


Train: 12000 images
Val:   2000 images


In [ ]:
train_ds.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="train",
    classes=TOP100_CLASSES,
)

val_ds.export(
    export_dir=EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="val",
    classes=TOP100_CLASSES,
)

yaml_path = os.path.join(EXPORT_DIR, "dataset.yaml")
print(open(yaml_path).read())

 100% |█████████████| 12000/12000 [4.3m elapsed, 0s remaining, 46.6 samples/s]      


INFO:eta.core.utils: 100% |█████████████| 12000/12000 [4.3m elapsed, 0s remaining, 46.6 samples/s]      


Directory '/content/drive/MyDrive/yolo_top100_rpi/dataset' already exists; export will be merged with existing files


 100% |███████████████| 2000/2000 [46.0s elapsed, 0s remaining, 46.9 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 2000/2000 [46.0s elapsed, 0s remaining, 46.9 samples/s]      


names:
  0: Clothing
  1: Man
  2: Tree
  3: Human face
  4: Person
  5: Woman
  6: Footwear
  7: Window
  8: Flower
  9: Wheel
  10: Plant
  11: Car
  12: Human hair
  13: Human arm
  14: Human head
  15: Girl
  16: Building
  17: Human body
  18: Mammal
  19: House
  20: Chair
  21: Tire
  22: Suit
  23: Fashion accessory
  24: Food
  25: Boy
  26: Table
  27: Skyscraper
  28: Land vehicle
  29: Boat
  30: Jeans
  31: Human eye
  32: Human hand
  33: Human leg
  34: Toy
  35: Tower
  36: Human nose
  37: Bicycle wheel
  38: Glasses
  39: Dress
  40: Vehicle
  41: Bird
  42: Sports equipment
  43: Street light
  44: Human mouth
  45: Palm tree
  46: Book
  47: Tableware
  48: Drink
  49: Bottle
  50: Bicycle
  51: Furniture
  52: Snack
  53: Sculpture
  54: Flag
  55: Dog
  56: Dessert
  57: Microphone
  58: Fruit
  59: Jacket
  60: Guitar
  61: Fast food
  62: Drum
  63: Sunglasses
  64: Poster
  65: Fish
  66: Baked goods
  67: Shelf
  68: Houseplant
  69: Flowerpot
  70: Airplane
 

In [ ]:
import torch
from ultralytics import YOLO

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

model = YOLO("yolo26n.pt")

model.train(
    data=yaml_path,
    epochs=60,
    imgsz=416,
    batch=32,
    device=0 if torch.cuda.is_available() else "cpu",
    optimizer="AdamW",
    lr0=3e-4,
    lrf=0.01,
    warmup_epochs=3,
    mosaic=1.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    patience=15,
    project=os.path.join(ROOT, "runs"),
    name="yolo26n_top100_rpi",
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA: True
Device: NVIDIA A100-SXM4-40GB
Ultralytics 8.4.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_top100_rpi/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchsc

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7819fa4c7f80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018, 

In [ ]:
best_pt  = os.path.join(ROOT, "runs", "yolo26n_top100_rpi", "weights", "best.pt")
final_pt = os.path.join(ROOT, "yolo26n_top100_rpi_final.pt")

shutil.copy(best_pt, final_pt)
print("Saved:", final_pt)

model_check = YOLO(final_pt)
print(f"Classes: {len(model_check.names)}")
print(model_check.names)

Saved: /content/drive/MyDrive/yolo_top100_rpi/yolo26n_top100_rpi_final.pt
Classes: 100
{0: 'Clothing', 1: 'Man', 2: 'Tree', 3: 'Human face', 4: 'Person', 5: 'Woman', 6: 'Footwear', 7: 'Window', 8: 'Flower', 9: 'Wheel', 10: 'Plant', 11: 'Car', 12: 'Human hair', 13: 'Human arm', 14: 'Human head', 15: 'Girl', 16: 'Building', 17: 'Human body', 18: 'Mammal', 19: 'House', 20: 'Chair', 21: 'Tire', 22: 'Suit', 23: 'Fashion accessory', 24: 'Food', 25: 'Boy', 26: 'Table', 27: 'Skyscraper', 28: 'Land vehicle', 29: 'Boat', 30: 'Jeans', 31: 'Human eye', 32: 'Human hand', 33: 'Human leg', 34: 'Toy', 35: 'Tower', 36: 'Human nose', 37: 'Bicycle wheel', 38: 'Glasses', 39: 'Dress', 40: 'Vehicle', 41: 'Bird', 42: 'Sports equipment', 43: 'Street light', 44: 'Human mouth', 45: 'Palm tree', 46: 'Book', 47: 'Tableware', 48: 'Drink', 49: 'Bottle', 50: 'Bicycle', 51: 'Furniture', 52: 'Snack', 53: 'Sculpture', 54: 'Flag', 55: 'Dog', 56: 'Dessert', 57: 'Microphone', 58: 'Fruit', 59: 'Jacket', 60: 'Guitar', 61: '

In [ ]:
val_results = model_check.val(
    data=yaml_path,
    split="val",
    imgsz=416,
    batch=32,
    device=0 if torch.cuda.is_available() else "cpu",
)

print(f"mAP50:     {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95:  {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")
print(f"Precision: {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall:    {val_results.results_dict['metrics/recall(B)']:.4f}")